<a href="https://colab.research.google.com/github/aniketsable586-ops/Project-3/blob/main/Project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
##Project 3 -Machine Learning for Fatigue Lifetime Prediction of Structural Steels ##

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

##Import data into collab and data cleaning##

df = pd.read_excel('NIMS_FATIGUE.xlsx')
#print(df.info)            ##information about data
x = df["Quenching temp"].mean() ## Calculate the mean of column
df.fillna({"Quenching temp": x}, inplace=True) ## replace blank cell with mean value
y= df["Tempering temp"].mean()
df.fillna({"Tempering temp": y}, inplace=True)
k= df["Normalizing temp"].mean()
df.fillna({"Normalizing temp": k}, inplace=True)
a= df["Upper yield stress"].mean()
df.fillna({"Upper yield stress": a}, inplace=True)
b= df["True fracture stress"].mean()
df.fillna({"True fracture stress": b}, inplace=True)

print(df.duplicated().sum())  ## Number of duplicated rows in the dataframe
df.drop_duplicates(inplace = True) ## Remove duplicate rows
#print(df.info)  ## Data frame size after removing the duplicate rows
print("Type of data: {}".format(type([df,'data'])))

##check data readiness like missing values etc.##

missing_counts = df.isnull().sum()
print("Missing values per column:\n", missing_counts)



##identifying redundant columns##

# 1. Isolate the specific columns you suspect are redundant
strength_cols = [
    'Upper yield stress',
    '0.2% proof stress ',
    'Tensile strength',
    'True fracture stress',
    'Elongation',
    'Reduction of area',
    'Vickers hardness'
]

# 2. Calculate the correlation matrix
corr_matrix = df[strength_cols].corr()

# 3. Visualize it using a heatmap
#plt.figure(figsize=(10, 8))
# annot=True shows the exact numbers, cmap colors the scale from blue (-1) to red (1)
#sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
#plt.title("Correlation Matrix of Mechanical Properties")
#plt.tight_layout()
#plt.show()

##Drop Heat and DS no.PDF column , 0.2% proof stress and vickers hardness from df##
df = df.drop(columns=['Heat','DS No. PDF','0.2% proof stress ','Vickers hardness'])

# Feature Engineering
df['Stress_Ratio'] = df['σa'] / df['Tensile strength']
df['Log_σa'] = np.log10(df['σa']) # Base 10 is standard for S-N curves
##split dataframe into Train and test ##
X = df.drop('Nf', axis=1)
y = df['Nf']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape: {}".format(X_train.shape))
print("y_train shape: {}".format(y_train.shape))
print("X_test shape: {}".format(X_test.shape))
print("y_test shape: {}".format(y_test.shape))



##Scaler and hot encoding in singl step parallely##

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Separate your columns by name
numeric_cols = ['Reduction Ratio', 'dA', 'dB','dC','Normalizing temp','Quenching temp','Tempering temp','Upper yield stress','Tensile strength','True fracture stress','Elongation','Reduction of area','σa']
categorical_cols = ['Testing Material','Type of test']

# 2. Build the transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),                     # Scales numbers
        ('cat', OneHotEncoder(sparse_output=False), categorical_cols) # Encodes text
    ]
).set_output(transform="pandas")

# 3. Fit on train, transform both
X_train_final = preprocessor.fit_transform(X_train)
X_test_final = preprocessor.transform(X_test)


##Select Model for ML## ##Train the model####evaluate the model performance##
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Initialize your base Random Forest model
base_rf = RandomForestRegressor(random_state=42)

# 2. Wrap it in the Target Regressor
# func=np.log1p takes the log of y_train during training
# inverse_func=np.expm1 reverses the log when you call .predict()
log_model = TransformedTargetRegressor(
    regressor=base_rf,
    func=np.log1p,
    inverse_func=np.expm1
)

# 3. Train the model
# You pass in the normal y_train; the wrapper transforms it automatically behind the scenes
log_model.fit(X_train_final, y_train)

# 4. Make predictions
# The wrapper automatically converts the outputs back to your massive original numbers
y_pred_log = log_model.predict(X_test_final)

# 5. Evaluate the model using the normal y_test and the reverted predictions
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2_log = r2_score(y_test, y_pred_log)

print("--- Log-Transformed Random Forest ---")
print(f"Mean Absolute Error (MAE): {mae_log:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_log:.2f}")
print(f"R-squared (R²): {r2_log:.2f}")

##Fine Tuning##
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# 1. Define the grid of settings you want to test
# The double underscore 'regressor__' targets the Random Forest inside the wrapper
param_distributions = {
    'regressor__n_estimators': [100, 200, 300, 500],          # Number of trees
    'regressor__max_depth': [None, 10, 20, 30, 40],           # Maximum depth of each tree
    'regressor__min_samples_split': [2, 5, 10, 20],           # Minimum rows required to split a node
    'regressor__min_samples_leaf': [1, 2, 4, 8],              # Minimum rows required at the end of a branch
    'regressor__max_features': ['sqrt', 'log2', 1.0]          # Number of features considered per split
}

# 2. Set up the Randomized Search
# cv=5 uses 5-fold cross-validation (crucial for your 2,500-row dataset)
# n_iter=20 means it will randomly pick 20 different combinations to test
random_search = RandomizedSearchCV(
    estimator=log_model, # Your existing TransformedTargetRegressor
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1 # Uses all your computer's processors to run faster
)

# 3. Run the search on your training data
# This will train 20 combinations * 5 folds = 100 separate models
print("Searching for the best hyperparameters. This may take a minute...")
random_search.fit(X_train_final, y_train)

# 4. Extract the winning model
best_model = random_search.best_estimator_

# 5. Evaluate the winning model on your unseen test data
y_pred_best = best_model.predict(X_test_final)

mae_best = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best = r2_score(y_test, y_pred_best)

print("--- Tuned Random Forest Evaluation ---")
print(f"Best Parameters: {random_search.best_params_}")
print(f"Mean Absolute Error (MAE): {mae_best:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_best:.2f}")
print(f"R-squared (R²): {r2_best:.2f}")











371
Type of data: <class 'list'>
Missing values per column:
 Heat                    0
Testing Material        0
DS No. PDF              0
Reduction Ratio         0
dA                      0
dB                      0
dC                      0
Normalizing temp        0
Quenching temp          0
Tempering temp          0
Upper yield stress      0
0.2% proof stress       0
Tensile strength        0
True fracture stress    0
Elongation              0
Reduction of area       0
Vickers hardness        0
Type of test            0
σa                      0
Nf                      0
dtype: int64
X_train shape: (1937, 17)
y_train shape: (1937,)
X_test shape: (485, 17)
y_test shape: (485,)
--- Log-Transformed Random Forest ---
Mean Absolute Error (MAE): 44622304.19
Root Mean Squared Error (RMSE): 402994881.03
R-squared (R²): 0.47
Searching for the best hyperparameters. This may take a minute...
--- Tuned Random Forest Evaluation ---
Best Parameters: {'regressor__n_estimators': 500, 'regressor__mi